# ZotVision — CNN-ViT Hybrid Classifier
**EfficientNet-B4 + Google ViT** for 4-class firefighter hazard detection  
Classes: `null` · `hazard` · `person` · `both`

Run this notebook end-to-end in **Google Colab (GPU)**. Each run is saved to its own uniquely-named folder in Google Drive (e.g. `iter1_a4f8c2/`) so runs never overwrite each other.

## 1 · Install dependencies

In [ ]:
!pip install -q efficientnet_pytorch transformers kagglehub seaborn scikit-learn

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Root ZotVision folder — iteration sub-folders are created automatically
GDRIVE_ROOT = '/content/drive/MyDrive/ZotVision'

import os
os.makedirs(GDRIVE_ROOT, exist_ok=True)
print(f'Google Drive root: {GDRIVE_ROOT}')

## 3 · Kaggle credentials & dataset download

In [ ]:
# Upload your kaggle.json here OR set env vars directly
# from google.colab import files
# files.upload()  # upload kaggle.json then uncomment the two lines below
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Alternatively — paste credentials inline (DO NOT commit this cell with real tokens)
import os, json
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY']      = 'your_api_key'
print('Kaggle credentials configured.')

## 4 · Imports & config

In [ ]:
"""
CNN-ViT Hybrid Model: EfficientNet + Google ViT
Output: 4-class classification → ['none', 'hazard', 'person', 'both']
CLS token is extracted from the ViT's last hidden state for classification.
"""

import hashlib
import json
import os
import random
import shutil
import time

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from efficientnet_pytorch import EfficientNet
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import ViTModel, ViTConfig

# ── Paths ──
KAGGLE_DATASET = 'varshvijay05/firefighter-hazard-dataset'
DATASET_DIR    = '/content/dataset'
IMAGES_DIR     = os.path.join(DATASET_DIR, 'images')
LABELS_FILE    = os.path.join(DATASET_DIR, 'results', 'labels.txt')
RESULTS_DIR    = os.path.join(DATASET_DIR, 'results')

# ── Labels ──
NUM_CLASSES  = 4
LABEL_MAP    = {'null': 0, 'hazard': 1, 'person': 2, 'both': 3}
ID_TO_LABEL  = {0: 'null', 1: 'hazard', 2: 'person', 3: 'both'}
CLASS_NAMES  = [ID_TO_LABEL[i] for i in range(NUM_CLASSES)]

# ── Training defaults ──
BATCH_SIZE   = 16
NUM_EPOCHS   = 20
LR           = 3e-4
IMG_SIZE     = 224
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Google Drive root (set in cell 2; re-declared here for standalone use) ──
try:
    GDRIVE_ROOT
except NameError:
    GDRIVE_ROOT = '/content/drive/MyDrive/ZotVision'

print(f'Device: {DEVICE}')

## 5 · Dataset

In [ ]:
def ensure_dataset():
    if os.path.isdir(IMAGES_DIR) and os.path.isfile(LABELS_FILE):
        print('[INFO] Dataset already present.')
        return
    print(f'[INFO] Downloading dataset from Kaggle: {KAGGLE_DATASET}')
    import kagglehub
    path = kagglehub.dataset_download(KAGGLE_DATASET)
    src_images  = os.path.join(path, 'images')
    src_results = os.path.join(path, 'results')
    if os.path.isdir(src_images):
        os.makedirs(IMAGES_DIR, exist_ok=True)
        for f in os.listdir(src_images):
            shutil.copy2(os.path.join(src_images, f), IMAGES_DIR)
    if os.path.isdir(src_results):
        os.makedirs(os.path.dirname(LABELS_FILE), exist_ok=True)
        for f in os.listdir(src_results):
            shutil.copy2(os.path.join(src_results, f), os.path.dirname(LABELS_FILE))
    print(f'[INFO] Dataset downloaded to {DATASET_DIR}')


ensure_dataset()

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


def load_samples(labels_file=LABELS_FILE, images_dir=IMAGES_DIR):
    """Parse labels.txt → list of (image_path, label_idx)."""
    samples = []
    with open(labels_file) as f:
        for i, line in enumerate(f):
            label_name = line.strip()
            if not label_name or label_name not in LABEL_MAP:
                continue
            img_path = os.path.join(images_dir, f'{i+1}.jpg')
            if os.path.exists(img_path):
                samples.append((img_path, LABEL_MAP[label_name]))
    return samples


def get_transforms(train):
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=0.2),
        ])
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


# Build 80/20 train/val split (fixed seed for reproducibility)
all_samples = load_samples()
random.seed(42)
random.shuffle(all_samples)
split = int(0.8 * len(all_samples))
train_samples = all_samples[:split]
val_samples   = all_samples[split:]

print(f'Total: {len(all_samples)}  |  Train: {len(train_samples)}  |  Val: {len(val_samples)}')

## 6 · Model — CNN-ViT Hybrid

In [ ]:
class CNNViTHybrid(nn.Module):
    """
    1. EfficientNet-B4  →  spatial feature map (C×H×W)
    2. Patch projection  →  sequence of patch tokens  (N×D)
    3. Prepend learnable CLS token
    4. Add positional embeddings
    5. Google ViT transformer encoder
    6. Extract CLS token  →  (B, D)
    7. MLP classifier head  →  4 logits
    """

    def __init__(
        self,
        efficientnet_variant='efficientnet-b4',
        vit_hidden_size=768,
        vit_num_layers=6,
        vit_num_heads=12,
        num_classes=NUM_CLASSES,
        dropout=0.3,
    ):
        super().__init__()

        self.cnn = EfficientNet.from_pretrained(efficientnet_variant)
        cnn_out_channels = self.cnn._conv_head.out_channels
        self.cnn._avg_pooling = nn.Identity()
        self.cnn._dropout     = nn.Identity()
        self.cnn._fc          = nn.Identity()

        for name, param in self.cnn.named_parameters():
            if not any(k in name for k in ['_blocks.28', '_blocks.29', '_blocks.30', '_blocks.31', '_conv_head']):
                param.requires_grad = False

        self.patch_proj = nn.Conv2d(cnn_out_channels, vit_hidden_size, kernel_size=1)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, vit_hidden_size))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        expected_patches = 49
        self.pos_embed = nn.Parameter(torch.zeros(1, expected_patches + 1, vit_hidden_size))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.vit_hidden_size = vit_hidden_size

        vit_config = ViTConfig(
            hidden_size=vit_hidden_size,
            num_hidden_layers=vit_num_layers,
            num_attention_heads=vit_num_heads,
            intermediate_size=vit_hidden_size * 4,
            hidden_dropout_prob=dropout,
            attention_probs_dropout_prob=dropout,
            image_size=IMG_SIZE,
            patch_size=16,
            num_channels=3,
        )
        self.vit_encoder = ViTModel(vit_config)

        self.classifier = nn.Sequential(
            nn.LayerNorm(vit_hidden_size),
            nn.Linear(vit_hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        B    = x.size(0)
        feat = self.cnn.extract_features(x)
        feat = self.patch_proj(feat)
        N    = feat.shape[2] * feat.shape[3]
        feat = feat.flatten(2).transpose(1, 2)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        tokens     = torch.cat([cls_tokens, feat], dim=1)

        if self.pos_embed.size(1) != N + 1:
            pos = torch.nn.functional.interpolate(
                self.pos_embed.transpose(1, 2), size=N + 1, mode='linear', align_corners=False
            ).transpose(1, 2)
        else:
            pos = self.pos_embed
        tokens = tokens + pos

        vit_out     = self.vit_encoder.encoder(tokens)
        last_hidden = self.vit_encoder.layernorm(vit_out.last_hidden_state)
        cls_out     = last_hidden[:, 0, :]
        return self.classifier(cls_out)

## 7 · Training utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_per_class(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        logits = model(imgs.to(device))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

print('Training utilities defined.')

## 8 · Per-label accuracy & confusion heatmap

In [ ]:
def compute_per_class_accuracy(preds, labels, class_names=CLASS_NAMES):
    """
    Prints classification report and displays a styled DataFrame of per-label
    accuracy (recall).  Returns a dict {label: accuracy}.
    """
    report_dict = classification_report(
        labels, preds, target_names=class_names, digits=4, output_dict=True
    )
    print(classification_report(labels, preds, target_names=class_names, digits=4))

    per_class_acc = {cls: round(report_dict[cls]['recall'], 4) for cls in class_names}

    df = pd.DataFrame(
        [
            {
                'Label':     cls,
                'Accuracy':  per_class_acc[cls],
                'Precision': round(report_dict[cls]['precision'], 4),
                'F1-score':  round(report_dict[cls]['f1-score'], 4),
                'Support':   int(report_dict[cls]['support']),
            }
            for cls in class_names
        ]
    ).set_index('Label')

    display(df.style.background_gradient(cmap='YlGn', subset=['Accuracy', 'F1-score']))
    return per_class_acc


def plot_confusion_heatmap(preds, labels, class_names=CLASS_NAMES, save_path=None):
    cm      = confusion_matrix(labels, preds, labels=range(len(class_names)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('Confusion Matrix (counts)')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Oranges',
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title('Confusion Matrix (normalized — diagonal = per-class accuracy)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f'Heatmap saved → {save_path}')
    plt.show()

print('Metrics helpers defined.')

## 9 · Google Drive helpers
Each call to `save_to_gdrive()` creates a new uniquely-named folder  
`MyDrive/ZotVision/iter<N>_<hash>/` and an `iterations.json` index at the root so the counter never resets.

In [ ]:
def _next_iter_folder(gdrive_root=GDRIVE_ROOT):
    """
    Returns a unique folder path for this run, e.g.
      .../ZotVision/iter3_a4f8c2
    An iterations.json index in gdrive_root tracks the counter so folders
    never collide across sessions, even if old ones are deleted.
    """
    index_path = os.path.join(gdrive_root, 'iterations.json')
    if os.path.exists(index_path):
        with open(index_path) as f:
            index = json.load(f)
    else:
        index = {'count': 0, 'runs': []}

    index['count'] += 1
    run_num = index['count']

    # 6-char hash from run number + timestamp — short but collision-proof
    short_hash = hashlib.sha1(f'{run_num}-{time.time()}'.encode()).hexdigest()[:6]
    folder_name = f'iter{run_num}_{short_hash}'
    folder_path = os.path.join(gdrive_root, folder_name)

    index['runs'].append({'iter': run_num, 'hash': short_hash, 'folder': folder_name})
    os.makedirs(gdrive_root, exist_ok=True)
    with open(index_path, 'w') as f:
        json.dump(index, f, indent=2)

    return folder_path


def save_to_gdrive(results_dir, gdrive_root=GDRIVE_ROOT):
    """
    Copy model weights, heatmaps, and result files into a new iteration
    folder on Google Drive.  Folder structure:

      MyDrive/ZotVision/
        iterations.json          ← run index (never deleted)
        iter1_a4f8c2/
          model_weights.pth
          model_run0.pth  model_run1.pth  ...
          heatmap_run0.png  heatmap_run1.png  ...
          run_comparison.png
          hyperparam_results.json
        iter2_9d3e71/
          ...
    """
    iter_dir = _next_iter_folder(gdrive_root)
    os.makedirs(iter_dir, exist_ok=True)
    print(f'\n[GDRIVE] Saving to: {iter_dir}')

    for fname in ['model_weights.pth', 'hyperparam_results.json', 'run_comparison.png']:
        src = os.path.join(results_dir, fname)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(iter_dir, fname))
            print(f'  ✓ {fname}')

    for fname in sorted(os.listdir(results_dir)):
        if fname.startswith(('model_run', 'heatmap_run')):
            shutil.copy2(os.path.join(results_dir, fname), os.path.join(iter_dir, fname))
            print(f'  ✓ {fname}')

    print(f'[GDRIVE] Done.')
    return iter_dir

print('Google Drive helpers defined.')

## 10 · Hyperparameter grid

In [ ]:
HYPERPARAM_GRID = [
    {'lr': 3e-4, 'dropout': 0.1, 'weight_decay': 1e-3, 'batch_size': 16, 'num_epochs': 20},
    {'lr': 1e-4, 'dropout': 0.2, 'weight_decay': 1e-4, 'batch_size': 16, 'num_epochs': 20},
    {'lr': 5e-4, 'dropout': 0.3, 'weight_decay': 1e-3, 'batch_size': 32, 'num_epochs': 20},
    {'lr': 3e-4, 'dropout': 0.2, 'weight_decay': 1e-2, 'batch_size':  8, 'num_epochs': 20},
    {'lr': 1e-3, 'dropout': 0.1, 'weight_decay': 1e-4, 'batch_size': 16, 'num_epochs': 15},
]

pd.DataFrame(HYPERPARAM_GRID)

## 11 · Training loop (all configs)

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)


def train_config(config, train_samples, val_samples, run_id, results_dir=RESULTS_DIR):
    print(f"\n{'='*60}")
    print(f'Run {run_id}: {config}')
    print(f"{'='*60}")

    train_ds = CustomDataset(train_samples, get_transforms(train=True))
    val_ds   = CustomDataset(val_samples,   get_transforms(train=False))
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    model = CNNViTHybrid(
        efficientnet_variant='efficientnet-b4',
        vit_hidden_size=768,
        vit_num_layers=6,
        vit_num_heads=12,
        num_classes=NUM_CLASSES,
        dropout=config['dropout'],
    ).to(DEVICE)

    optimizer     = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['num_epochs'])
    class_weights = torch.tensor([1.0, 1.2, 1.0, 2.5]).to(DEVICE)
    criterion     = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

    best_val_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    weights_path = os.path.join(results_dir, f'model_run{run_id}.pth')

    for epoch in range(1, config['num_epochs'] + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss,   val_acc   = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch:03d}/{config['num_epochs']} | "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), weights_path)

    # ── Per-class evaluation ──
    model.load_state_dict(torch.load(weights_path, weights_only=True))
    preds, labels = evaluate_per_class(model, val_loader, DEVICE)
    per_class_acc = compute_per_class_accuracy(preds, labels)
    plot_confusion_heatmap(preds, labels, save_path=os.path.join(results_dir, f'heatmap_run{run_id}.png'))

    return {
        'run_id': run_id,
        'config': config,
        'best_val_acc': best_val_acc,
        'per_class_acc': per_class_acc,
        'history': history,
    }

In [ ]:
all_results = []
for i, config in enumerate(HYPERPARAM_GRID):
    result = train_config(config, train_samples, val_samples, run_id=i)
    all_results.append(result)

## 12 · Results & comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for r in all_results:
    axes[0].plot(r['history']['val_acc'], label=f"run{r['run_id']} (acc={r['best_val_acc']:.3f})")
axes[0].set_title('Validation Accuracy per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=8)

run_labels = [f"run{r['run_id']}" for r in all_results]
accs       = [r['best_val_acc'] for r in all_results]
bars       = axes[1].bar(run_labels, accs, color=sns.color_palette('viridis', len(all_results)))
axes[1].set_title('Best Validation Accuracy per Config')
axes[1].set_ylabel('Accuracy')
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 f'{acc:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'run_comparison.png'), dpi=150)
plt.show()

In [ ]:
# ── Per-label accuracy across all runs ──
rows = []
for r in all_results:
    row = {'Run': f"run{r['run_id']} (val_acc={r['best_val_acc']:.4f})"}
    row.update(r['per_class_acc'])
    rows.append(row)

df_per_class = pd.DataFrame(rows).set_index('Run')
display(df_per_class.style.background_gradient(cmap='YlGn').format('{:.4f}'))

In [ ]:
# ── Save best model as model_weights.pth ──
best = max(all_results, key=lambda r: r['best_val_acc'])
print(f"BEST: run{best['run_id']}  val_acc={best['best_val_acc']:.4f}")
print(f"Config: {best['config']}")
print(f"Per-class accuracy: {best['per_class_acc']}")

best_src = os.path.join(RESULTS_DIR, f"model_run{best['run_id']}.pth")
best_dst = os.path.join(RESULTS_DIR, 'model_weights.pth')
shutil.copy2(best_src, best_dst)
print(f'Best weights → {best_dst}')

# ── Persist summary JSON ──
summary = [
    {'run_id': r['run_id'], 'config': r['config'],
     'best_val_acc': r['best_val_acc'], 'per_class_acc': r['per_class_acc']}
    for r in all_results
]
with open(os.path.join(RESULTS_DIR, 'hyperparam_results.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Results JSON → {os.path.join(RESULTS_DIR, 'hyperparam_results.json')}")

## 13 · Save everything to Google Drive
Creates a new `iter<N>_<hash>/` folder so this run never overwrites a previous one.

In [ ]:
saved_to = save_to_gdrive(RESULTS_DIR)
print(f'\nView your Drive folder: {saved_to}')

## 14 · Inference on a single image

In [ ]:
@torch.no_grad()
def predict(model, image_path, device=DEVICE):
    """Run inference on a single image; returns label string."""
    transform = get_transforms(train=False)
    img    = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    model.eval()
    logits = model(tensor)
    pred   = logits.argmax(1).item()
    return ID_TO_LABEL[pred]


# Example usage (replace with any image path)
# best_model = CNNViTHybrid().to(DEVICE)
# best_model.load_state_dict(torch.load(best_dst, weights_only=True))
# print(predict(best_model, '/path/to/image.jpg'))